In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import glob
import sys

In [ ]:
def clean_treatment_name(name):
    name = str(name)
    name = name.replace("\n", " ")
    name = name.replace("\xa0", " ")  # non-breaking space
    name = re.sub(r"\s+", " ", name).strip()
    return name


def read_seahorse_excel(file_path, sheet_name):
    # Read Excel:
    # - row 0 = merged condition headers
    # - row 1 = replicate headers or blank second header row
    df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=[0, 1])

    # Forward-fill the merged top-row condition names across columns
    level0 = pd.Series(df_raw.columns.get_level_values(0)).ffill()
    level1 = pd.Series(df_raw.columns.get_level_values(1))

    # Build clean flat column names
    new_cols = []
    rep_count = {}

    for cond, sub in zip(level0, level1):
        cond = clean_treatment_name(cond)

        # Handle Time / Cycle columns
        if "Time" in cond:
            new_cols.append("Time (minutes)")
            continue
        if "Cycle" in cond:
            new_cols.append("Cycle")
            continue

        # Number replicates within each condition
        rep_count.setdefault(cond, 0)
        rep_count[cond] += 1
        new_cols.append(f"{cond}_rep{rep_count[cond]}")

    df_raw.columns = new_cols
    return df_raw.copy()


def calculate_seahorse_from_cycles(
    df,
    baseline_window,
    oligomycin_window,
    fccp_window,
    rotaa_window,
    cycle_col="Cycle",
    time_col="Time (minutes)",
    use_fccp_max=True
):
    results = []

    replicate_cols = [c for c in df.columns if c not in [cycle_col, time_col]]

    for col in replicate_cols:
        sub = df[[cycle_col, col]].dropna()

        basal_vals = sub.loc[
            sub[cycle_col].between(baseline_window[0], baseline_window[1]),
            col
        ]

        olig_vals = sub.loc[
            sub[cycle_col].between(oligomycin_window[0], oligomycin_window[1]),
            col
        ]

        fccp_vals = sub.loc[
            sub[cycle_col].between(fccp_window[0], fccp_window[1]),
            col
        ]

        rotaa_vals = sub.loc[
            sub[cycle_col].between(rotaa_window[0], rotaa_window[1]),
            col
        ]

        basal_ocr = basal_vals.mean()
        olig_ocr = olig_vals.mean()
        fccp_ocr = fccp_vals.max() if use_fccp_max else fccp_vals.mean()
        rotaa_ocr = rotaa_vals.mean()

        non_mito = rotaa_ocr
        basal_resp = basal_ocr - non_mito
        atp_linked = basal_ocr - olig_ocr
        proton_leak = olig_ocr - non_mito
        maximal_resp = fccp_ocr - non_mito
        spare_capacity = maximal_resp - basal_resp
        coupling_eff = atp_linked / basal_resp if basal_resp != 0 else np.nan
        rcr_like = maximal_resp / proton_leak if proton_leak != 0 else np.nan

        treatment = clean_treatment_name(col.rsplit("_rep", 1)[0])

        results.append({
            "Treatment": treatment,
            "Replicate": col,
            "Basal_OCR": basal_ocr,
            "Oligomycin_OCR": olig_ocr,
            "FCCP_OCR": fccp_ocr,
            "RotAA_OCR": rotaa_ocr,
            "Non_mitochondrial_respiration": non_mito,
            "Basal_respiration": basal_resp,
            "ATP_linked_respiration": atp_linked,
            "Proton_leak": proton_leak,
            "Maximal_respiration": maximal_resp,
            "Spare_respiratory_capacity": spare_capacity,
            "Coupling_efficiency": coupling_eff,
            "RCR_like_metric": rcr_like
        })

    return pd.DataFrame(results)


def calculate_ecar_from_cycles(
    df,
    baseline_window,
    oligomycin_window,
    cycle_col="Cycle",
    time_col="Time (minutes)"
):
    results = []

    replicate_cols = [c for c in df.columns if c not in [cycle_col, time_col]]

    for col in replicate_cols:
        sub = df[[cycle_col, col]].dropna()

        basal_vals = sub.loc[
            sub[cycle_col].between(baseline_window[0], baseline_window[1]),
            col
        ]

        olig_vals = sub.loc[
            sub[cycle_col].between(oligomycin_window[0], oligomycin_window[1]),
            col
        ]

        basal_ecar = basal_vals.mean()
        olig_ecar = olig_vals.mean()
        compensatory_glycolysis = olig_ecar - basal_ecar

        treatment = clean_treatment_name(col.rsplit("_rep", 1)[0])

        results.append({
            "Treatment": treatment,
            "Replicate": col,
            "Basal_ECAR": basal_ecar,
            "ECAR_after_oligomycin": olig_ecar,
            "Compensatory_glycolysis": compensatory_glycolysis
        })

    return pd.DataFrame(results)


# ----------------------------
# Input Excel file (edit this path each run)
# ----------------------------
input_file = "NWD-19_SKQ1/N1/NWD19_SKQ1_raw_adjusted_cleaned.xlsx"
print(f"Reading: {input_file}")

# ----------------------------
# Read OCR and ECAR sheets
# ----------------------------
# Adjust sheet names here if yours differ
OCR_SHEET = "OCR"
ECAR_SHEET = "ECAR"

df_ocr = read_seahorse_excel(input_file, sheet_name=OCR_SHEET)
df_ecar = read_seahorse_excel(input_file, sheet_name=ECAR_SHEET)

# ----------------------------
# Cycle windows (shared between OCR and ECAR — same wells, same cycles)
# Edit these once per experiment based on your injection schedule.
# ----------------------------
BASELINE_WINDOW   = (1, 3)
OLIGOMYCIN_WINDOW = (4, 6)
FCCP_WINDOW       = (7, 9)
ROTAA_WINDOW      = (10, 12)

# ----------------------------
# Calculate OCR metrics
# ----------------------------
params_ocr = calculate_seahorse_from_cycles(
    df_ocr,
    cycle_col="Cycle",
    baseline_window=BASELINE_WINDOW,
    oligomycin_window=OLIGOMYCIN_WINDOW,
    fccp_window=FCCP_WINDOW,
    rotaa_window=ROTAA_WINDOW,
)

# ----------------------------
# Calculate ECAR metrics
# (only baseline and oligomycin windows are needed — Compensatory_glycolysis
#  is the only ECAR-derived metric and it doesn't depend on FCCP or Rot/AA)
# ----------------------------
params_ecar = calculate_ecar_from_cycles(
    df_ecar,
    cycle_col="Cycle",
    baseline_window=BASELINE_WINDOW,
    oligomycin_window=OLIGOMYCIN_WINDOW,
)

# ----------------------------
# Merge OCR + ECAR metrics
# ----------------------------
params = pd.merge(
    params_ocr,
    params_ecar,
    on=["Treatment", "Replicate"],
    how="outer"
)

print(params)

# Optional: summary by treatment
summary = params.groupby("Treatment", as_index=False).agg({
    "Non_mitochondrial_respiration": ["mean", "std"],
    "Basal_respiration": ["mean", "std"],
    "Basal_OCR": ["mean", "std"],
    "ATP_linked_respiration": ["mean", "std"],
    "Proton_leak": ["mean", "std"],
    "Maximal_respiration": ["mean", "std"],
    "Spare_respiratory_capacity": ["mean", "std"],
    "Coupling_efficiency": ["mean", "std"],
    "RCR_like_metric": ["mean", "std"],
    "Basal_ECAR": ["mean", "std"],
    "ECAR_after_oligomycin": ["mean", "std"],
    "Compensatory_glycolysis": ["mean", "std"],
})

print(summary)

# -------------------------------------------------------------------
# Export one CSV per metric, with treatments as side-by-side columns
# -------------------------------------------------------------------

csv_metrics = [
    "Non_mitochondrial_respiration",
    "Basal_respiration",
    "Basal_OCR",
    "ATP_linked_respiration",
    "Proton_leak",
    "Maximal_respiration",
    "Spare_respiratory_capacity",
    "Coupling_efficiency",
    "RCR_like_metric",
    "Basal_ECAR",
    "ECAR_after_oligomycin",
    "Compensatory_glycolysis"
]

# Auto-extract treatment order from the Excel headers (preserves original order).
# df_ocr columns look like "Media only_rep1", "Media only_rep2", "DMSO_rep1", ...
# plus "Cycle" and "Time (minutes)" which we exclude.
treatment_order = []
for col in df_ocr.columns:
    if col in ("Cycle", "Time (minutes)"):
        continue
    treatment = clean_treatment_name(col.rsplit("_rep", 1)[0])
    if treatment not in treatment_order:
        treatment_order.append(treatment)

print(f"Treatment order: {treatment_order}")

# Output folder for CSVs (placed alongside the input Excel file)
csv_output_dir = Path(input_file).resolve().parent / "seahorse_metric_csv"
csv_output_dir.mkdir(exist_ok=True)

for metric in csv_metrics:
    export_df = pd.DataFrame()

    for treatment in treatment_order:
        vals = params.loc[params["Treatment"] == treatment, metric].reset_index(drop=True)
        export_df[treatment] = vals

    export_path = csv_output_dir / f"{metric}.csv"
    export_df.to_csv(export_path, index=False)

    print(f"Saved: {export_path}")

# -------------------------------------------------------------------
# Plot one PNG per metric (bar with replicate dots, ordered by Excel)
# -------------------------------------------------------------------

plot_metrics = [
    "Basal_OCR",
    "Oligomycin_OCR",
    "FCCP_OCR",
    "RotAA_OCR",
    "Non_mitochondrial_respiration",
    "Basal_respiration",
    "ATP_linked_respiration",
    "Proton_leak",
    "Maximal_respiration",
    "Spare_respiratory_capacity",
    "Coupling_efficiency",
    "RCR_like_metric",
    "Basal_ECAR",
    "ECAR_after_oligomycin",
    "Compensatory_glycolysis"
]

# Title and subtitle for each metric
metric_labels = {
    "Basal_OCR": (
        "Basal OCR",
        "Oxygen consumption rate under baseline conditions prior to mitochondrial inhibition."
    ),
    "Oligomycin_OCR": (
        "Oligomycin OCR",
        "Residual oxygen consumption following ATP synthase inhibition, reflecting non-ATP-linked respiration."
    ),
    "FCCP_OCR": (
        "FCCP OCR",
        "Maximum oxygen consumption achieved under uncoupled conditions, representing peak electron transport chain activity."
    ),
    "RotAA_OCR": (
        "Rotenone/Antimycin A OCR",
        "Residual oxygen consumption after complete mitochondrial inhibition, representing non-mitochondrial respiration."
    ),
    "Non_mitochondrial_respiration": (
        "Non-mitochondrial respiration",
        "Oxygen consumption independent of the mitochondrial electron transport chain, measured after rotenone/antimycin A."
    ),
    "Basal_respiration": (
        "Basal respiration",
        "Mitochondrial oxygen consumption under basal conditions, calculated as baseline OCR minus non-mitochondrial respiration."
    ),
    "ATP_linked_respiration": (
        "ATP-linked respiration",
        "Oxygen consumption coupled to ATP production, calculated as the decrease in OCR following oligomycin."
    ),
    "Proton_leak": (
        "Proton leak",
        "Oxygen consumption not coupled to ATP synthesis, reflecting mitochondrial membrane leak or uncoupling."
    ),
    "Maximal_respiration": (
        "Maximal respiration",
        "Maximum mitochondrial respiratory capacity under uncoupled conditions, corrected for non-mitochondrial respiration."
    ),
    "Spare_respiratory_capacity": (
        "Spare respiratory capacity",
        "Difference between maximal and basal respiration, indicating the ability of mitochondria to respond to increased energy demand."
    ),
    "Coupling_efficiency": (
        "Coupling efficiency",
        "Fraction of basal mitochondrial respiration used for ATP production."
    ),
    "RCR_like_metric": (
        "RCR-like metric",
        "Ratio of maximal respiration to proton leak, representing an approximation of mitochondrial coupling efficiency in intact cells."
    ),
    "Basal_ECAR": (
        "Basal ECAR",
        "Extracellular acidification rate under baseline conditions, reflecting glycolytic activity."
    ),
    "ECAR_after_oligomycin": (
        "ECAR after oligomycin",
        "Extracellular acidification rate following ATP synthase inhibition, reflecting maximal compensatory glycolysis."
    ),
    "Compensatory_glycolysis": (
        "Compensatory glycolysis",
        "Increase in ECAR following oligomycin, indicating glycolytic response to mitochondrial ATP synthesis blockade."
    ),
}

# Output folder for PNGs (placed alongside the input Excel file)
plot_output_dir = Path(input_file).resolve().parent / "seahorse_metric_plots"
plot_output_dir.mkdir(exist_ok=True)

for metric in plot_metrics:

    summary_plot = params.groupby("Treatment").agg(
        mean=(metric, "mean"),
        sem=(metric, "sem")
    ).reindex(treatment_order).reset_index()

    fig, ax = plt.subplots(figsize=(6, 5))
    x = np.arange(len(summary_plot))

    # Bar plot: black outline, clear fill
    ax.bar(
        x,
        summary_plot["mean"],
        yerr=summary_plot["sem"],
        capsize=5,
        edgecolor="black",
        facecolor="none",
        linewidth=1.5
    )

    # Overlay replicate dots in black
    for i, treatment in enumerate(treatment_order):
        vals = params.loc[params["Treatment"] == treatment, metric].dropna()
        jitter = np.random.normal(0, 0.05, size=len(vals))

        ax.scatter(
            np.full(len(vals), i) + jitter,
            vals,
            s=30,
            color="black"
        )

    # Axis formatting
    ax.set_xticks(x)
    ax.set_xticklabels(treatment_order, rotation=45, ha="right")
    ax.set_xlabel("Condition")
    ax.set_ylabel(metric.replace("_", " "))

    # Title + subtitle
    title, subtitle = metric_labels[metric]
    ax.set_title(title, fontsize=14, pad=20)
    ax.text(
        0.5, 1.02, subtitle,
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9
    )

    plt.tight_layout()

    # Save plot
    save_path = plot_output_dir / f"{metric}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved: {save_path}")